In [2]:
# Run these commands in Anaconda Prompt or a terminal:
# 1. Navigate to the repository folder:
#    cd data_processing
# 2. Create the environment:
#    conda env create -f environment.yml
# 3. Activate it:
#    conda activate cs424env
# 4. Open the notebook:
#    jupyter notebook visulas.ipynb
# 5. Use the kernel for cs424env and run all cells from top to bottom.

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import altair as alt


In [3]:
# Load our data
data = pd.read_csv("../Updated_Dataset.csv")
locations = pd.read_csv("locations.csv")
fixes = pd.read_csv("fixes.csv")

In [4]:
# Blank out the values we know are wrong
for visit, column in zip(fixes["visit_id"], fixes["column"]):
    data.loc[data["visit_id"] == visit, column] = None

# Recalculate these instead of trusting the spreadsheet formulas
data["available_chairs"] = data["total_chairs"] - data["occupied_chairs"]
data["available_tables"] = data["total_tables"] - data["occupied_tables"]
data["chair_occupancy_pct"] = data["occupied_chairs"] / data["total_chairs"] * 100
data["table_occupancy_pct"] = data["occupied_tables"] / data["total_tables"] * 100
data = data.drop(columns=["chair_occupancy_rate", "table_occupancy_rate"])

# Print any visits that are wrong
problems = data[data["occupied_tables"] > data["occupied_chairs"]]
print("Visits with more occupied tables than chairs:", list(problems["visit_id"]))

# Add new columns
data = data.merge(locations, on="location")
data["session_id"] = data["date"] + " " + data["time_of_day"]
data["table_chair_gap"] = data["table_occupancy_pct"] - data["chair_occupancy_pct"]
data["people_per_table"] = data["occupied_chairs"] / data["occupied_tables"]
data["has_open_table"] = (data["available_tables"] > 0).astype(float)
data.loc[data["available_tables"].isna(), "has_open_table"] = None
data.round(2).to_csv("../processed_data/observations_clean.csv", index=False)

Visits with more occupied tables than chairs: []


In [5]:
# Q1: average occupancy for each lounge at each time of day
q1 = data.groupby(["location_label", "time_of_day"])[["chair_occupancy_pct", "table_occupancy_pct"]].mean()
q1.round(2).to_csv("../processed_data/q1_by_time_of_day.csv")

# Q2: the same lounge, day and time of day, on a rainy day vs a sunny day
q2 = data.pivot_table(index=["location_label", "day", "time_of_day"], columns="weather",
    values="chair_occupancy_pct")[["Rain", "Sunny"]].dropna()
q2.round(2).to_csv("../processed_data/q2_rain_vs_sun.csv")

# Q4 and our new question: open seats, open tables and people per table for each lounge
q4 = data.groupby("location_label")[["chair_occupancy_pct", "table_occupancy_pct", "available_chairs",
    "has_open_table", "people_per_table"]].mean()
q4.round(2).sort_values("chair_occupancy_pct").to_csv("../processed_data/q4_by_location.csv")

# How many visits we have for each day and time of day
days = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]
times = ["Morning", "Midday", "Afternoon"]
coverage = pd.crosstab(data["day"], data["time_of_day"]).reindex(index=days, columns=times, fill_value=0)
coverage.to_csv("../processed_data/coverage.csv")
periods = ["Morning", "Midday", "Afternoon"]

In [6]:
coverage_data = coverage.rename_axis(
    index="day", columns="time_of_day"
).stack().reset_index(name="visits")

base = alt.Chart(coverage_data).encode(
    x=alt.X("time_of_day:O", sort=times, title="Time of day"),
    y=alt.Y("day:O", sort=days, title="Weekday")
)

cells = base.mark_rect().encode(
    color=alt.Color("visits:Q", scale=alt.Scale(scheme="blues"), title="Visits")
)

labels = base.mark_text().encode(
    text="visits:Q",
    color=alt.condition(
        alt.datum.visits > coverage_data["visits"].max() / 2,
        alt.value("white"),
        alt.value("black")
    )
)

(cells + labels).properties(
    width=400,
    height=250,
    title="Number of visits"
)

alt.LayerChart(...)

In [7]:
weekly_data = pd.concat([
    data.assign(series="All observations"),
    data[data["time_of_day"] == "Morning"].assign(series="Morning only")
])

alt.Chart(weekly_data).mark_line(point=True).encode(
    x=alt.X("semester_week:O", title="Semester week"),
    y=alt.Y(
        "mean(chair_occupancy_pct):Q",
        title="Average chair occupancy (%)",
        scale=alt.Scale(domain=[0, 100])
    ),
    color=alt.Color("series:N", title="Observations included")
).properties(
    width=600,
    height=350,
    title="Weekly chair occupancy"
)

alt.Chart(...)

In [8]:
subset = data[
    (data["location_label"] == "West Terrace")
    & (data["time_of_day"] == "Morning")
    & (data["weather"].isin(["Sunny", "Rain"]))
]

alt.Chart(subset).mark_bar().encode(
    x=alt.X("weather:N", title="Weather", sort=["Sunny", "Rain"]),
    y=alt.Y(
        "mean(chair_occupancy_pct):Q",
        title="Average chair occupancy (%)",
        scale=alt.Scale(domain=[0, 100])
    ),
    tooltip=[
        alt.Tooltip("mean(chair_occupancy_pct):Q", format=".1f"),
        alt.Tooltip("valid(chair_occupancy_pct):Q", title="Valid visits")
    ]
).properties(
    width=350,
    height=300,
    title="West Terrace: morning occupancy by weather"
)

alt.Chart(...)

In [9]:
subset = data[data["location_label"] == "Montgomery Lounge"]

alt.Chart(subset).mark_point(filled=True, size=70).encode(
    x=alt.X(
        "chair_occupancy_pct:Q",
        title="Chair occupancy (%)",
        scale=alt.Scale(domain=[0, 100])
    ),
    y=alt.Y(
        "table_occupancy_pct:Q",
        title="Table occupancy (%)",
        scale=alt.Scale(domain=[0, 100])
    ),
    color=alt.Color("time_of_day:N", title="Time of day", sort=periods),
    shape=alt.Shape("time_of_day:N", title="Time of day", sort=periods),
    tooltip=["visit_id", "chair_occupancy_pct", "table_occupancy_pct"]
).properties(
    width=400,
    height=400,
    title="Montgomery Lounge: tables versus chairs"
)

alt.Chart(...)

In [10]:
alt.Chart(data).mark_line(point=True).encode(
    x=alt.X(
        "time_of_day:O",
        title="Time of day",
        sort=periods,
        axis=alt.Axis(labelAngle=0)
    ),
    y=alt.Y(
        "mean(chair_occupancy_pct):Q",
        title="Average chair occupancy (%)",
        scale=alt.Scale(domain=[0, 100])
    ),
    color=alt.Color("location_label:N", title="Study area")
).properties(
    width=600,
    height=350,
    title="Chair occupancy throughout the day"
)

alt.Chart(...)

In [11]:
alt.Chart(data).mark_line(point=True).encode(
    x=alt.X(
        "time_of_day:O",
        title="Time of day",
        sort=periods,
        axis=alt.Axis(labelAngle=0)
    ),
    y=alt.Y(
        "mean(available_tables):Q",
        title="Average empty tables",
        scale=alt.Scale(zero=True)
    ),
    color=alt.Color("location_label:N", title="Study area")
).properties(
    width=600,
    height=350,
    title="Empty tables throughout the day"
)

alt.Chart(...)

In [12]:
alt.Chart(data).mark_bar().encode(
    x=alt.X(
        "location_label:N",
        title="Study area",
        axis=alt.Axis(labelAngle=-35)
    ),
    xOffset=alt.XOffset("time_of_day:N", sort=periods),
    y=alt.Y(
        "mean(has_open_table):Q",
        title="Visits with an empty table",
        scale=alt.Scale(domain=[0, 1]),
        axis=alt.Axis(format=".0%")
    ),
    color=alt.Color("time_of_day:N", title="Time of day", sort=periods),
    tooltip=[
        "location_label",
        "time_of_day",
        alt.Tooltip("mean(has_open_table):Q", title="Frequency", format=".0%"),
        alt.Tooltip("valid(has_open_table):Q", title="Valid visits")
    ]
).properties(
    width=700,
    height=350,
    title="How often was an empty table available?"
)

alt.Chart(...)

In [13]:
occupancy_comparison = data.melt(
    id_vars=["location_label"],
    value_vars=["chair_occupancy_pct", "table_occupancy_pct"],
    var_name="occupancy_type",
    value_name="occupancy"
)

occupancy_comparison["occupancy_type"] = (
    occupancy_comparison["occupancy_type"].replace({
        "chair_occupancy_pct": "Chairs",
        "table_occupancy_pct": "Tables"
    })
)

bar_chart = alt.Chart(occupancy_comparison).mark_bar().encode(
    x=alt.X(
        "location_label:N",
        title="Study area",
        axis=alt.Axis(labelAngle=-35)
    ),
    xOffset="occupancy_type:N",
    y=alt.Y(
        "mean(occupancy):Q",
        title="Average occupancy (%)",
        scale=alt.Scale(domain=[0, 100])
    ),
    color=alt.Color("occupancy_type:N", title="Occupancy type"),
    tooltip=[
        "location_label:N",
        "occupancy_type:N",
        alt.Tooltip("mean(occupancy):Q", title="Average (%)", format=".1f"),
        alt.Tooltip("valid(occupancy):Q", title="Valid visits")
    ]
).properties(
    width=700,
    height=350,
    title="Average table and chair occupancy by location"
)

bar_chart

alt.Chart(...)

In [14]:
heatmap_data = (
    data.groupby(["location_label", "time_of_day"])["chair_occupancy_pct"]
    .agg(average_occupancy="mean", observations="count")
    .reset_index()
)

base = alt.Chart(heatmap_data).encode(
    x=alt.X(
        "time_of_day:O",
        sort=periods,
        title="Time of day",
        axis=alt.Axis(labelAngle=0)
    ),
    y=alt.Y("location_label:N", title="Study area")
)

cells = base.mark_rect().encode(
    color=alt.Color(
        "average_occupancy:Q",
        title="Occupancy (%)",
        scale=alt.Scale(domain=[0, 100], scheme="blues")
    ),
    tooltip=[
        "location_label:N",
        "time_of_day:O",
        alt.Tooltip("average_occupancy:Q", title="Average (%)", format=".1f"),
        alt.Tooltip("observations:Q", title="Valid visits")
    ]
)

labels = base.mark_text().encode(
    text=alt.Text("average_occupancy:Q", format=".1f"),
    color=alt.condition(
        alt.datum.average_occupancy > 60,
        alt.value("white"),
        alt.value("black")
    )
)

heatmap_chart = (cells + labels).properties(
    width=450,
    height=350,
    title="Average chair occupancy by location and time"
)

heatmap_chart

alt.LayerChart(...)

In [17]:
scatter_chart = alt.Chart(data).mark_point(
    filled=True,
    opacity=0.65,
    size=55
).encode(
    x=alt.X(
        "chair_occupancy_pct:Q",
        title="Chair occupancy (%)",
        scale=alt.Scale(domain=[0, 100])
    ),
    y=alt.Y(
        "table_occupancy_pct:Q",
        title="Table occupancy (%)",
        scale=alt.Scale(domain=[0, 100])
    ),
    color=alt.Color("time_of_day:N", title="Time of day", sort=periods),
    tooltip=[
        "visit_id:N",
        "location_label:N",
        "time_of_day:N",
        alt.Tooltip("chair_occupancy_pct:Q", title="Chairs (%)", format=".1f"),
        alt.Tooltip("table_occupancy_pct:Q", title="Tables (%)", format=".1f")
    ]
).properties(
    width=200,
    height=200
).facet(
    facet=alt.Facet("location_label:N", title=None),
    columns=3
).properties(
    title="Table and chair occupancy across individual visits"
)

scatter_chart

alt.FacetChart(...)

In [19]:
locations = sorted(data["location_label"].dropna().unique())

selected_location = alt.param(
    value=locations[0],
    bind=alt.binding_select(
        options=locations,
        name="Study area: "
    )
)

scatter_chart = (
    alt.Chart(data)
    .mark_point(filled=True, size=80, opacity=0.75)
    .encode(
        x=alt.X(
            "chair_occupancy_pct:Q",
            title="Chair occupancy (%)",
            scale=alt.Scale(domain=[0, 100])
        ),
        y=alt.Y(
            "table_occupancy_pct:Q",
            title="Table occupancy (%)",
            scale=alt.Scale(domain=[0, 100])
        ),
        color=alt.Color(
            "time_of_day:N",
            title="Time of day",
            sort=periods
        ),
        shape=alt.Shape(
            "weather:N",
            title="Weather"
        ),
        tooltip=[
            alt.Tooltip("visit_id:N", title="Visit"),
            alt.Tooltip("location_label:N", title="Study area"),
            alt.Tooltip("date:N", title="Date"),
            alt.Tooltip("time:N", title="Time"),
            alt.Tooltip("time_of_day:N", title="Time of day"),
            alt.Tooltip("weather:N", title="Weather"),
            alt.Tooltip(
                "chair_occupancy_pct:Q",
                title="Chair occupancy (%)",
                format=".1f"
            ),
            alt.Tooltip(
                "table_occupancy_pct:Q",
                title="Table occupancy (%)",
                format=".1f"
            )
        ]
    )
    .add_params(selected_location)
    .transform_filter(
        alt.datum.location_label == selected_location
    )
    .properties(
        width=450,
        height=450,
        title="Table versus chair occupancy by study area"
    )
)

scatter_chart

alt.Chart(...)